# Machine Learning-Based Energy Consumption Prediction in 5G Mobile Communication Networks
### End-to-End Walkthrough Notebook

**Theme:** Machine Learning x Network Communication  
**Author:**
- Sonika Reddy (@sonikareddy24)

---

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.config import STATION_PROFILES, REQUIRED_COLUMNS
from src.data_generator import generate_synthetic_5g_data
from src.preprocessing import preprocess_data
from src.features import build_features, chronological_train_val_test_split, FEATURE_COLUMNS, TARGET_COLUMN
from src.models import train_models
from src.evaluation import evaluate_models_leaderboard, compute_naive_baseline
from src.forecasting import forecast_energy, detect_peak_windows
from src.optimizer import simulate_energy_savings

print("Modules imported successfully!")

## 1. Data Generation
Generate 12 months (8,760 hours) of realistic 5G base station telemetry across 5 distinct cell profiles.

In [ ]:
raw_df = generate_synthetic_5g_data(num_hours=8760, save_path=None)
print(f"Generated dataset shape: {raw_df.shape}")
raw_df.head()

## 2. Preprocessing & IQR Outlier Handling
Perform timestamp sorting, time-based missing value interpolation, and IQR fence capping.

In [ ]:
clean_df = preprocess_data(raw_df)
print(f"Missing values remaining: {clean_df.isna().sum().sum()}")
print(f"Outlier readings bounded: {clean_df['is_outlier'].sum()}")

## 3. Causal Feature Engineering & Chronological Split
Generate shifted lags, shifted rolling statistics, cyclical calendar features, and split 70% Train, 15% Val, 15% Test.

In [ ]:
featured_df = build_features(clean_df, drop_warmup=True)
train_df, val_df, test_df = chronological_train_val_test_split(featured_df)

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
print(f"Max Train: {train_df['timestamp'].max()} < Min Test: {test_df['timestamp'].min()}")

## 4. Machine Learning Model Suite & Evaluation Leaderboard
Train Linear Regression, Random Forest, and XGBoost/GradientBoosting.

In [ ]:
X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df[TARGET_COLUMN]

models, times, fallback = train_models(X_train, y_train, mode="Fast")
leaderboard, best_name, _ = evaluate_models_leaderboard(models, X_test, y_test, times)

naive_m = compute_naive_baseline(test_df)
print("=== Model Leaderboard ===")
display(leaderboard)
print(f"Naive Baseline RMSE: {naive_m['RMSE']:.4f} kWh (R²: {naive_m['R2']:.4f})")

## 5. Multi-Step Recursive Forecasting & High-Consumption Peak Detection
Forecast 48 hours ahead for Urban base station BS_01 and flag high-energy periods.

In [ ]:
best_model = models[best_name]
bs01_history = clean_df[clean_df["station_id"] == "BS_01"]

fc_df = forecast_energy(best_model, bs01_history, horizon_hours=48)
annotated_fc, peak_windows = detect_peak_windows(fc_df, percentile=90.0)

print(f"Flagged {annotated_fc['is_peak'].sum()} peak hours across {len(peak_windows)} contiguous windows.")
display(peak_windows)

## 6. What-If Energy Saver Lab
Simulate putting 33% carrier capacity to sleep during off-peak hours (PRB load ≤ 30%).

In [ ]:
sim_res = simulate_energy_savings(
    fc_df,
    best_model,
    carrier_sleep_pct=33.3,
    traffic_offload_pct=15.0,
    prb_threshold_pct=30.0,
)

print(sim_res["recommendation"])
print(f"Energy Saved: {sim_res['total_kwh_saved']:.2f} kWh ({sim_res['pct_saved']:.1f}% reduction)")
print(f"Cost Saved: ${sim_res['cost_saved_usd']:.2f} USD")
print(f"CO2 Avoided: {sim_res['co2_saved_kg']:.2f} kg")